# Chapter 11: Stage 7 — Building Tool-Using AI Agents (Function Calling, Memory, Validation & Human-in-the-Loop Approval)

**Assalam-o-Alaikum!** Ab hum **Stage 7: AI Agents** seekhenge!

### 1. Chatbot vs AI Agent Mein Kya Farq Hai?
- **Simple Chatbot:** Sirf text par baat karta hai — na woh aap ka database check kar sakta hai, na student ke marks predict kar sakta hai, aur na refund initiate kar sakta hai.
- **AI Agent (`Brain + Tools + Memory + Guardrails`):** User ki request samajhta hai, faisla karta hai ke **kaunsa Python Tool/Function call karna hai** (`predict_student_marks_tool`, `search_policy_tool`, `issue_refund_tool`), tool ke inputs validate karta hai, aur **sensitive actions (maslan `> 10,000 PKR` refund)** ke liye **Human Approval** mangta hai!

In [1]:
import json
import joblib
import pandas as pd

### Step 1 — Define Verified Tools with Input Validation & Human-in-the-Loop Guardrails

In [2]:
student_pipe = joblib.load("models/student_marks_pipeline.pkl")
with open("datasets/company_knowledge_base.json", "r", encoding="utf-8") as f:
    kb_docs = json.load(f)

def tool_predict_marks(study_hours_daily: float, attendance_pct: float, previous_score: float, sleep_hours: float = 7.0, practice_tests: int = 3):
    if not (0.0 <= study_hours_daily <= 16.0):
        raise ValueError("study_hours_daily must be between 0 and 16.")
    df_in = pd.DataFrame([{
        "study_hours_daily": study_hours_daily,
        "attendance_pct": attendance_pct,
        "previous_score": previous_score,
        "sleep_hours": sleep_hours,
        "practice_tests": practice_tests,
    }])
    score = float(student_pipe.predict(df_in)[0])
    return {"predicted_marks": round(min(100.0, max(0.0, score)), 2)}

def tool_issue_refund(order_id: int, amount_pkr: float, human_approved: bool = False):
    if amount_pkr > 10000 and not human_approved:
        return {
            "status": "PENDING_HUMAN_APPROVAL",
            "message": f"Refund of {amount_pkr} PKR for Order #{order_id} exceeds 10,000 PKR limit (POL-04). Human manager approval required!",
        }
    return {
        "status": "APPROVED",
        "message": f"Refund of {amount_pkr} PKR for Order #{order_id} processed successfully.",
    }

class PracticalAIAgent:
    def __init__(self):
        self.memory = []

    def run(self, user_goal: str, params: dict):
        self.memory.append({"role": "user", "goal": user_goal, "params": params})
        if "marks" in user_goal.lower():
            result = tool_predict_marks(**params)
            tool_used = "tool_predict_marks"
        elif "refund" in user_goal.lower():
            result = tool_issue_refund(**params)
            tool_used = "tool_issue_refund"
        else:
            result = {"error": "No matching tool found."}
            tool_used = None
        response = {"goal": user_goal, "tool_called": tool_used, "output": result}
        self.memory.append({"role": "agent", "response": response})
        return response

agent = PracticalAIAgent()
print("Step 1 (Marks Tool):", agent.run("Predict student marks", {"study_hours_daily": 7.0, "attendance_pct": 90.0, "previous_score": 82.0}))
print("Step 2 (Small Refund):", agent.run("Process refund", {"order_id": 105, "amount_pkr": 2500}))
print("Step 3 (Large Refund Guardrail):", agent.run("Process refund", {"order_id": 992, "amount_pkr": 25000, "human_approved": False}))

Step 1 (Marks Tool): {'goal': 'Predict student marks', 'tool_called': 'tool_predict_marks', 'output': {'predicted_marks': 97.06}}
Step 2 (Small Refund): {'goal': 'Process refund', 'tool_called': 'tool_issue_refund', 'output': {'status': 'APPROVED', 'message': 'Refund of 2500 PKR for Order #105 processed successfully.'}}
Step 3 (Large Refund Guardrail): {'goal': 'Process refund', 'tool_called': 'tool_issue_refund', 'output': {'status': 'PENDING_HUMAN_APPROVAL', 'message': 'Refund of 25000 PKR for Order #992 exceeds 10,000 PKR limit (POL-04). Human manager approval required!'}}


### Step 2 — Inspect Agent Memory & Multi-Step Execution Trace
Ek AI Agent apni `self.memory` mein har step (`User Goal` $ightarrow$ `Tool Called` $ightarrow$ `Tool Output`) ka record rakhta hai taake multi-step workflows aur debugging asaan ho:

In [3]:
memory_df = pd.DataFrame([
    m["response"] for m in agent.memory if m["role"] == "agent"
])
memory_df

,goal,tool_called,output
0,Predict student marks,tool_predict_marks,{'predicted_marks': 97.06}
1,Process refund,tool_issue_refund,"{'status': 'APPROVED', 'message': 'Refund of 2500 PKR for Order #105 processed successfully.'}"
2,Process refund,tool_issue_refund,"{'status': 'PENDING_HUMAN_APPROVAL', 'message': 'Refund of 25000 PKR for Order #992 exceeds 10,000 PKR limit (POL-04). Human manager approval required!'}"


### Step 3 — Error Handling & Tool Input Validation Test
Agar koi user ya LLM tool ko ghalat parameter (`study_hours_daily = 35.0` hours in a single day!) bhej de, toh Agent ka **Tool Validation Guardrail** usay foran rok deta hai:

In [4]:
try:
    agent.run("Predict student marks", {"study_hours_daily": 35.0, "attendance_pct": 90.0, "previous_score": 80.0})
except ValueError as err:
    print("Agent Guardrail Caught Invalid Tool Input:", err)

Agent Guardrail Caught Invalid Tool Input: study_hours_daily must be between 0 and 16.


### Step 4 — Human-in-the-Loop Approval Workflow
Jab `amount_pkr = 25000` (`> 10,000 PKR`) ka refund aaya, toh Agent ne khud paise transfer nahi kiye balkay `PENDING_HUMAN_APPROVAL` return kiya. Ab jab Human Manager `human_approved=True` dega tabhi action mukammal hoga:

In [5]:
approved_res = agent.run("Process refund", {"order_id": 992, "amount_pkr": 25000, "human_approved": True})
print("After Human Manager Approval:", approved_res)

After Human Manager Approval: {'goal': 'Process refund', 'tool_called': 'tool_issue_refund', 'output': {'status': 'APPROVED', 'message': 'Refund of 25000 PKR for Order #992 processed successfully.'}}


### Chapter 11 Revision Quiz
1. **Chatbot aur AI Agent mein sab se bada farq kya hai?** Chatbot sirf text generate karta hai, jabke AI Agent tools/functions call kar ke external systems mein kaam (actions) kar sakta hai.
2. **Sensitive actions (Refunds, Deleting Data, Sending Emails) mein Human-in-the-Loop kyun zaroori hai?** Taake AI ki kisi ghalti ya prompt injection se mali (financial) ya data ka nuqsan na ho.